# 🚀 Falcon 9 Powered Landing — From Free Fall to Suicide Burn

We build the simulation in three stages:
1. **Free Fall** — No thrust. The rocket tumbles and crashes. This is the "before" picture.
2. **Powered Descent** — Max thrust + PD attitude control. The suicide burn.
3. **Guided Landing** — Variable thrust + lateral steering. Land on the pad.

Each stage teaches something new. Run them in order.

---
## Stage 1: Free Fall — What Happens Without Control

The simplest case: a rocket falling under gravity with a slight initial tilt and spin.  
No engine, no control. Just Newton's laws.

**What to observe:**
- The center of mass follows a perfect parabola regardless of tilt (gravity acts on CG)
- The rocket rotates at constant angular velocity (no torque → no angular acceleration)
- It crashes in a few seconds

This is what would happen if the engine fails to reignite during landing.

In [ ]:
using ModelingToolkit, OrdinaryDiffEqTsit5, GLMakie
using ModelingToolkit: t_nounits as t, D_nounits as D

# ============================================================
# Stage 1: Free Fall — gravity only, no thrust
# ============================================================
#
# States: x, y (position), θ (tilt), vx, vy (velocity), ω (spin)
# Only force: gravity (downward)
# No torque → ω stays constant, θ grows linearly

@parameters g_ff
@variables x_ff(t) y_ff(t) θ_ff(t) vx_ff(t) vy_ff(t) ω_ff(t)

eqs_ff = [
    D(x_ff)  ~ vx_ff,
    D(y_ff)  ~ vy_ff,
    D(θ_ff)  ~ ω_ff,
    D(vx_ff) ~ 0.0,           # no horizontal force
    D(vy_ff) ~ -g_ff,         # gravity pulls down
    D(ω_ff)  ~ 0.0,           # no torque → constant spin rate
]

@mtkbuild freefall_sys = ODESystem(eqs_ff, t)

# --- Initial Conditions ---
# Rocket at 500m, falling at 80 m/s, drifting sideways,
# tilted 10° with a slow tumble of 3°/s
u0_ff = [
    x_ff  => 50.0,             # 50m to the right of the pad
    y_ff  => 500.0,            # 500m altitude
    θ_ff  => deg2rad(10.0),    # tilted 10° from vertical
    vx_ff => 10.0,             # drifting right at 10 m/s
    vy_ff => -80.0,            # falling at 80 m/s
    ω_ff  => deg2rad(3.0),     # tumbling at 3°/s
]

p_ff = [g_ff => 9.81]

# Stop when rocket hits the ground (y = 0)
# NOTE: MTK reorders states internally, so we find the index dynamically
y_ff_idx = findfirst(isequal(y_ff), unknowns(freefall_sys))
println("y_ff is at state index: ", y_ff_idx)

ground_hit_ff(u, t, integrator) = u[y_ff_idx]
cb_ff = ContinuousCallback(ground_hit_ff, terminate!)

# Solve (new MTK API: merge u0 and p into one dict)
prob_ff = ODEProblem(freefall_sys, merge(Dict(u0_ff), Dict(p_ff)), (0.0, 30.0))
sol_ff = solve(prob_ff, Tsit5(); callback=cb_ff, dtmax=0.05)

println("Impact time: ", round(sol_ff.t[end], digits=2), " s")
println("Impact velocity: ", round(sol_ff[vy_ff][end], digits=1), " m/s")
println("Final tilt: ", round(rad2deg(sol_ff[θ_ff][end]), digits=1), "°")
println("Lateral drift: ", round(sol_ff[x_ff][end], digits=1), " m")

In [ ]:
# --- Plot the free fall ---
fig1 = Figure(size=(1000, 700))

ax1a = Axis(fig1[1, 1], xlabel="Time (s)", ylabel="Altitude (m)",
            title="Altitude — Parabolic Fall")
lines!(ax1a, sol_ff.t, sol_ff[y_ff], color=:dodgerblue, linewidth=2)

ax1b = Axis(fig1[1, 2], xlabel="Time (s)", ylabel="Vertical Velocity (m/s)",
            title="Vertical Velocity — Linear Increase")
lines!(ax1b, sol_ff.t, sol_ff[vy_ff], color=:red, linewidth=2)

ax1c = Axis(fig1[2, 1], xlabel="Time (s)", ylabel="Tilt Angle (°)",
            title="Tilt — Constant Spin (No Torque)")
lines!(ax1c, sol_ff.t, rad2deg.(sol_ff[θ_ff]), color=:orange, linewidth=2)

ax1d = Axis(fig1[2, 2], xlabel="X Position (m)", ylabel="Y Position (m)",
            title="Trajectory — CG Parabola")
lines!(ax1d, sol_ff[x_ff], sol_ff[y_ff], color=:green, linewidth=2)
scatter!(ax1d, [sol_ff[x_ff][end]], [sol_ff[y_ff][end]], color=:red, markersize=15,
         label="Impact")
axislegend(ax1d)

fig1

### What did we learn?

- **Altitude**: parabolic — exactly what `y(t) = y₀ + vy₀·t − ½g·t²` predicts.
- **Tilt**: linearly increasing — with no torque, angular velocity is constant, so the angle just ramps.
- **Trajectory**: the CG traces a clean parabola. Tilt doesn't affect the CG path — only an external force can.

This is why control matters. Without thrust vector control, the rocket is a ballistic brick.

---
## Stage 2: Powered Descent — Suicide Burn + Attitude Control

Now we add the engine. The full physics:

| Equation | What it says |
|---|---|
| `m·v̇x = −T·sin(θ+δ)` | Thrust's horizontal component (depends on where the rocket+engine points) |
| `m·v̇y = T·cos(θ+δ) − m·g` | Thrust's vertical component minus gravity |
| `I·ω̇ = −T·L·sin(δ)` | Engine gimbal creates torque around the CG |
| `dm/dt = −T/(g₀·Isp)` | Burning fuel reduces mass |

**Control strategy:**
- **Attitude (inner loop):** PD controller on tilt angle θ → commands gimbal angle δ
  - `δ = Kp·θ + Kd·ω`, smoothly clamped to ±5° gimbal limit
  - This keeps the rocket vertical during descent
- **Thrust:** Constant at max thrust (845 kN). This IS the suicide burn — full throttle, timed to zero velocity at zero altitude.

**The key constraint:**  
Merlin engine can only throttle down to ~40%. At minimum throttle, thrust EXCEEDS the rocket's weight.  
**You cannot hover.** You must time the burn perfectly.

In [ ]:
# ============================================================
# Stage 2: Powered Descent — Full thrust + PD attitude control
# ============================================================

# --- Falcon 9 First Stage Parameters ---
const GRAVITY    = 9.81       # m/s²
const T_MAX      = 845_000.0  # N, single Merlin 1D max thrust
const ISP        = 282.0      # s, specific impulse at sea level
const G0         = 9.81       # m/s², for mass flow calculation
const L_ENGINE   = 15.0       # m, CG to engine distance
const M_DRY      = 22_200.0   # kg, dry mass of first stage
const M_FUEL     = 3_000.0    # kg, fuel reserved for landing burn
const M0         = M_DRY + M_FUEL  # total mass at ignition
const I_BODY     = (1/12) * M0 * 42.0^2  # kg·m², rod approximation (42m tall)
const DELTA_MAX  = deg2rad(5.0)  # max gimbal angle (rad)

# --- What altitude should we ignite? ---
# From kinematics: v² = 2·a·d, where a = T/m - g (net deceleration)
a_net = T_MAX / M0 - GRAVITY
vy0_burn = -80.0
h_ignite = vy0_burn^2 / (2 * a_net)
println("Net deceleration at max thrust: ", round(a_net, digits=1), " m/s²")
println("Ignition altitude for 80 m/s descent: ", round(h_ignite, digits=0), " m")
println("Moment of inertia: ", round(I_BODY, digits=0), " kg·m²")
println("Fuel budget → burn time ≈ ",
        round(M_FUEL / (T_MAX / (G0 * ISP)), digits=1), " s")

In [ ]:
# ============================================================
# Define the ODE System — Rocket dynamics + attitude control
# ============================================================
#
# 7 differential states: x, y, θ, vx, vy, ω, m
# 1 algebraic variable: δ (gimbal angle, computed by PD controller)

@parameters g_s2 T_s2 L_s2 I_s2 Isp_s2 g0_s2 Kp_att Kd_att δ_lim
@variables x_s2(t) y_s2(t) θ_s2(t) vx_s2(t) vy_s2(t) ω_s2(t) m_s2(t)
@variables δ_s2(t)  # gimbal angle (algebraic)

eqs_s2 = [
    # --- Kinematics ---
    D(x_s2)  ~ vx_s2,
    D(y_s2)  ~ vy_s2,
    D(θ_s2)  ~ ω_s2,

    # --- Translational dynamics (F = ma) ---
    D(vx_s2) ~ -(T_s2 / m_s2) * sin(θ_s2 + δ_s2),
    D(vy_s2) ~  (T_s2 / m_s2) * cos(θ_s2 + δ_s2) - g_s2,

    # --- Rotational dynamics (τ = Iα) ---
    D(ω_s2)  ~ -(T_s2 * L_s2 / I_s2) * sin(δ_s2),

    # --- Mass depletion ---
    D(m_s2)  ~ -T_s2 / (g0_s2 * Isp_s2),

    # --- Attitude Control (PD on tilt angle) ---
    # Smooth saturation via tanh keeps δ within gimbal limits
    δ_s2 ~ δ_lim * tanh((Kp_att * θ_s2 + Kd_att * ω_s2) / δ_lim),
]

@mtkbuild powered_sys = ODESystem(eqs_s2, t)

# --- Find state index for y (for ground callback) ---
y_s2_idx = findfirst(isequal(y_s2), unknowns(powered_sys))
println("y_s2 is at state index: ", y_s2_idx)

In [ ]:
# --- Solve the powered descent ---
u0_s2 = [
    x_s2  => 0.0,
    y_s2  => h_ignite * 0.98,    # slightly below kinematic estimate (mass depletion increases decel)
    θ_s2  => deg2rad(5.0),       # 5° initial tilt
    vx_s2 => 0.0,
    vy_s2 => vy0_burn,           # falling at 80 m/s
    ω_s2  => deg2rad(1.0),       # slight initial spin
    m_s2  => M0,
]

p_s2 = [
    g_s2   => GRAVITY,
    T_s2   => T_MAX,             # CONSTANT max thrust — suicide burn
    L_s2   => L_ENGINE,
    I_s2   => I_BODY,
    Isp_s2 => ISP,
    g0_s2  => G0,
    Kp_att => 1.5,
    Kd_att => 1.5,
    δ_lim  => DELTA_MAX,
]

ground_hit_s2(u, t, integrator) = u[y_s2_idx]
cb_s2 = ContinuousCallback(ground_hit_s2, terminate!)

prob_s2 = ODEProblem(powered_sys, merge(Dict(u0_s2), Dict(p_s2)), (0.0, 30.0))
sol_s2 = solve(prob_s2, Tsit5(); callback=cb_s2, dtmax=0.01)

println("--- Suicide Burn Results ---")
println("Burn time: ", round(sol_s2.t[end], digits=2), " s")
println("Landing velocity: ", round(sol_s2[vy_s2][end], digits=2), " m/s")
println("Final tilt: ", round(rad2deg(sol_s2[θ_s2][end]), digits=2), "°")
println("Fuel used: ", round(M0 - sol_s2[m_s2][end], digits=0), " kg")
println("Lateral drift: ", round(sol_s2[x_s2][end], digits=1), " m")

In [ ]:
# --- Plot the powered descent ---
fig2 = Figure(size=(1200, 800))

ax2a = Axis(fig2[1, 1], xlabel="Time (s)", ylabel="Altitude (m)",
            title="Altitude")
lines!(ax2a, sol_s2.t, sol_s2[y_s2], color=:dodgerblue, linewidth=2)
hlines!(ax2a, [0.0], color=:black, linestyle=:dash, label="Ground")
axislegend(ax2a)

ax2b = Axis(fig2[1, 2], xlabel="Time (s)", ylabel="Vertical Velocity (m/s)",
            title="Vertical Velocity")
lines!(ax2b, sol_s2.t, sol_s2[vy_s2], color=:red, linewidth=2)
hlines!(ax2b, [0.0], color=:black, linestyle=:dash, label="Zero")
axislegend(ax2b)

ax2c = Axis(fig2[1, 3], xlabel="Time (s)", ylabel="Tilt Angle (°)",
            title="Tilt (Attitude Control Active)")
lines!(ax2c, sol_s2.t, rad2deg.(sol_s2[θ_s2]), color=:orange, linewidth=2)
hlines!(ax2c, [0.0], color=:black, linestyle=:dash)

ax2d = Axis(fig2[2, 1], xlabel="Time (s)", ylabel="Gimbal Angle (°)",
            title="Gimbal Command δ")
lines!(ax2d, sol_s2.t, rad2deg.(sol_s2[δ_s2]), color=:purple, linewidth=2)
hlines!(ax2d, [rad2deg(DELTA_MAX), -rad2deg(DELTA_MAX)], color=:red,
        linestyle=:dash, label="±5° limit")
axislegend(ax2d)

ax2e = Axis(fig2[2, 2], xlabel="Time (s)", ylabel="Mass (kg)",
            title="Mass Depletion")
lines!(ax2e, sol_s2.t, sol_s2[m_s2], color=:green, linewidth=2)
hlines!(ax2e, [M_DRY], color=:red, linestyle=:dash, label="Dry mass")
axislegend(ax2e)

ax2f = Axis(fig2[2, 3], xlabel="X (m)", ylabel="Y (m)",
            title="Trajectory")
lines!(ax2f, sol_s2[x_s2], sol_s2[y_s2], color=:black, linewidth=2)
scatter!(ax2f, [0.0], [0.0], color=:red, markersize=15, marker=:star5,
         label="Landing Pad")
axislegend(ax2f)

fig2

### What to look for

- **Altitude**: should decrease to ~0 (if the ignition altitude was computed right).
- **Vertical velocity**: starts at −80 m/s and should approach 0 at landing. If it's still negative at y=0, the burn started too late. If it goes positive, too early.
- **Tilt**: the PD controller should drive θ → 0 within ~2-3 seconds. Watch the initial transient.
- **Gimbal**: expect an initial kick (correcting the 5° tilt), then settling near 0. Should stay within ±5° limits.
- **Mass**: drops linearly (constant thrust → constant fuel burn rate).

**Experiment:** Try changing `θ_s2 => deg2rad(15.0)` — a much bigger initial tilt. Does the controller still recover?

---
## Stage 3: Guided Landing — Throttle Control + Lateral Steering

The constant-thrust suicide burn from Stage 2 has a problem: it doesn't steer laterally, and the timing has to be exact. Real Falcon 9 uses **guidance laws** that modulate thrust and steer toward the pad.

We add two things:

### Thrust Guidance
Instead of constant thrust, we define a **desired velocity profile**:

`vy_desired(y) = −C · √y`

This says: at height `y`, you should be falling at speed `C·√y`. As altitude decreases, desired speed decreases. At y=0, desired speed is 0.

The thrust controller tracks this profile:

`T = m · (g + C²/2 + Kv · (vy_desired − vy))`

- `C²/2` is the **feedforward** — the deceleration the profile demands
- `Kv` is the **feedback** — corrects deviations

### Lateral Guidance
To steer toward x=0, we command a small tilt angle:

`θ_cmd = Kp_x · x + Kd_x · vx`

The attitude controller then tracks θ_cmd instead of 0.

In [ ]:
# ============================================================
# Stage 3: Guided Landing — variable thrust + lateral steering
# ============================================================

@parameters begin
    g_s3
    T_max_s3
    L_s3
    I_s3
    Isp_s3
    g0_s3
    Kp_att_s3
    Kd_att_s3
    δ_lim_s3
    C_guide
    Kv_guide
    Kp_lat
    Kd_lat
    θ_cmd_lim
end

@variables begin
    x_s3(t)
    y_s3(t)
    θ_s3(t)
    vx_s3(t)
    vy_s3(t)
    ω_s3(t)
    m_s3(t)
    vy_des_s3(t)
    T_raw_s3(t)
    T_pos_s3(t)
    T_s3(t)
    θ_cmd_s3(t)
    δ_s3(t)
end

# Softplus sharpness for thrust clamp
k_sat = 20.0

eqs_s3 = [
    # --- Kinematics ---
    D(x_s3)  ~ vx_s3,
    D(y_s3)  ~ vy_s3,
    D(θ_s3)  ~ ω_s3,

    # --- Translational dynamics ---
    D(vx_s3) ~ -(T_s3 / m_s3) * sin(θ_s3 + δ_s3),
    D(vy_s3) ~  (T_s3 / m_s3) * cos(θ_s3 + δ_s3) - g_s3,

    # --- Rotational dynamics ---
    D(ω_s3)  ~ -(T_s3 * L_s3 / I_s3) * sin(δ_s3),

    # --- Mass depletion ---
    D(m_s3)  ~ -T_s3 / (g0_s3 * Isp_s3),

    # --- Guidance: desired descent velocity profile ---
    vy_des_s3 ~ -C_guide * sqrt(max(y_s3, 0.01)),

    # --- Thrust command: feedforward + feedback ---
    T_raw_s3 ~ m_s3 * (g_s3 + C_guide^2 / 2.0 + Kv_guide * (vy_des_s3 - vy_s3)),

    # --- Smooth clamp [0, T_max] using softplus (two-stage) ---
    # Step 1: max(0, T_raw) via softplus lower bound
    T_pos_s3 ~ (T_max_s3 / k_sat) * log(1.0 + exp(k_sat * T_raw_s3 / T_max_s3)),
    # Step 2: min(T_max, T_pos) via softplus upper bound
    T_s3 ~ T_max_s3 - (T_max_s3 / k_sat) * log(1.0 + exp(k_sat * (T_max_s3 - T_pos_s3) / T_max_s3)),

    # --- Lateral guidance ---
    θ_cmd_s3 ~ θ_cmd_lim * tanh((Kp_lat * x_s3 + Kd_lat * vx_s3) / θ_cmd_lim),

    # --- Attitude control: PD on (θ - θ_cmd) ---
    δ_s3 ~ δ_lim_s3 * tanh((Kp_att_s3 * (θ_s3 - θ_cmd_s3) + Kd_att_s3 * ω_s3) / δ_lim_s3),
]

@mtkbuild guided_sys = ODESystem(eqs_s3, t)

y_s3_idx = findfirst(isequal(y_s3), unknowns(guided_sys))
println("y_s3 is at state index: ", y_s3_idx)

In [ ]:
# --- Solve the guided landing ---
u0_s3 = [
    x_s3  => 15.0,              # 15m offset from pad
    y_s3  => 500.0,             # 500m altitude
    θ_s3  => deg2rad(3.0),      # slight initial tilt
    vx_s3 => 3.0,               # small lateral drift
    vy_s3 => -50.0,             # falling at 50 m/s
    ω_s3  => 0.0,
    m_s3  => M0,
]

p_s3 = [
    g_s3       => GRAVITY,
    T_max_s3   => T_MAX,
    L_s3       => L_ENGINE,
    I_s3       => I_BODY,
    Isp_s3     => ISP,
    g0_s3      => G0,
    Kp_att_s3  => 1.5,
    Kd_att_s3  => 1.5,
    δ_lim_s3   => DELTA_MAX,
    C_guide    => 2.5,           # velocity profile steepness
    Kv_guide   => 2.0,           # thrust feedback gain
    Kp_lat     => 0.006,         # lateral position gain
    Kd_lat     => 0.04,          # lateral velocity gain
    θ_cmd_lim  => deg2rad(15.0), # max lateral tilt command
]

ground_hit_s3(u, t, integrator) = u[y_s3_idx]
cb_s3 = ContinuousCallback(ground_hit_s3, terminate!)

prob_s3 = ODEProblem(guided_sys, merge(Dict(u0_s3), Dict(p_s3)), (0.0, 60.0))
sol_s3 = solve(prob_s3, Tsit5(); callback=cb_s3, dtmax=0.01)

println("--- Guided Landing Results ---")
println("Total time: ", round(sol_s3.t[end], digits=2), " s")
println("Landing velocity: ", round(sol_s3[vy_s3][end], digits=2), " m/s")
println("Lateral position: ", round(sol_s3[x_s3][end], digits=1), " m")
println("Lateral velocity: ", round(sol_s3[vx_s3][end], digits=2), " m/s")
println("Final tilt: ", round(rad2deg(sol_s3[θ_s3][end]), digits=2), "°")
println("Fuel used: ", round(M0 - sol_s3[m_s3][end], digits=0), " kg of $M_FUEL available")

In [ ]:
# --- Plot the guided landing ---
fig3 = Figure(size=(1400, 900))

ax3a = Axis(fig3[1, 1], xlabel="Time (s)", ylabel="Altitude (m)",
            title="Altitude")
lines!(ax3a, sol_s3.t, sol_s3[y_s3], color=:dodgerblue, linewidth=2)

ax3b = Axis(fig3[1, 2], xlabel="Time (s)", ylabel="Velocity (m/s)",
            title="Vertical Velocity vs Desired")
lines!(ax3b, sol_s3.t, sol_s3[vy_s3], color=:red, linewidth=2, label="Actual vy")
lines!(ax3b, sol_s3.t, sol_s3[vy_des_s3], color=:blue, linewidth=2,
       linestyle=:dash, label="Desired vy")
hlines!(ax3b, [0.0], color=:black, linestyle=:dot)
axislegend(ax3b)

ax3c = Axis(fig3[1, 3], xlabel="Time (s)", ylabel="Thrust (kN)",
            title="Thrust Profile")
lines!(ax3c, sol_s3.t, sol_s3[T_s3] ./ 1000, color=:orange, linewidth=2)
hlines!(ax3c, [T_MAX/1000, 0.4*T_MAX/1000], color=:red, linestyle=:dash,
        label="Max / 40% min")
axislegend(ax3c)

ax3d = Axis(fig3[2, 1], xlabel="Time (s)", ylabel="Tilt (°)",
            title="Tilt & Lateral Command")
lines!(ax3d, sol_s3.t, rad2deg.(sol_s3[θ_s3]), color=:orange, linewidth=2,
       label="θ actual")
lines!(ax3d, sol_s3.t, rad2deg.(sol_s3[θ_cmd_s3]), color=:blue, linewidth=1.5,
       linestyle=:dash, label="θ commanded")
axislegend(ax3d)

ax3e = Axis(fig3[2, 2], xlabel="Time (s)", ylabel="Lateral Position (m)",
            title="Lateral Convergence to Pad")
lines!(ax3e, sol_s3.t, sol_s3[x_s3], color=:green, linewidth=2)
hlines!(ax3e, [0.0], color=:red, linestyle=:dash, label="Pad center")
axislegend(ax3e)

ax3f = Axis(fig3[2, 3], xlabel="X (m)", ylabel="Y (m)",
            title="Full Trajectory")
lines!(ax3f, sol_s3[x_s3], sol_s3[y_s3], color=:black, linewidth=2)
scatter!(ax3f, [0.0], [0.0], color=:red, markersize=20, marker=:star5,
         label="Pad")
scatter!(ax3f, [sol_s3[x_s3][1]], [sol_s3[y_s3][1]], color=:blue, markersize=12,
         label="Start")
axislegend(ax3f)

fig3

### What to look for in Stage 3

- **vy vs vy_desired**: Actual velocity should converge onto the desired profile.
- **Thrust**: Should start low or zero (coasting), then ramp up as engine ignites, then modulate.
- **Lateral position**: Should converge from 15m toward 0.
- **Trajectory**: Curved path from offset toward the pad.

**Tuning experiments:**
- `C_guide = 4.0` → more aggressive descent (faster velocity profile)
- `Kp_lat = 0.008` → more aggressive lateral steering
- `x_s3 => 100.0, vx_s3 => 15.0` → bigger offset, harder landing

---
## Stage 4: The Animation

A tracking camera follows the rocket down, zooming in as it descends. Live telemetry plots altitude, velocity, and thrust in real time.

In [ ]:
# ============================================================
# Resample solution + add hold frames
# ============================================================

N_sim = 500
t_sim = range(sol_s3.t[1], sol_s3.t[end], length=N_sim)

x_idx  = findfirst(isequal(x_s3), unknowns(guided_sys))
y_idx  = findfirst(isequal(y_s3), unknowns(guided_sys))
θ_idx  = findfirst(isequal(θ_s3), unknowns(guided_sys))

x_data  = [sol_s3(ti)[x_idx] for ti in t_sim]
y_data  = [sol_s3(ti)[y_idx] for ti in t_sim]
θ_data  = [sol_s3(ti)[θ_idx] for ti in t_sim]
vy_data = [sol_s3(ti, idxs=vy_s3) for ti in t_sim]
T_data  = [sol_s3(ti, idxs=T_s3) for ti in t_sim]

N_hold = 60  # 2 seconds at 30fps
x_all  = vcat(x_data, fill(x_data[end], N_hold))
y_all  = vcat(y_data, fill(max(y_data[end], 0.0), N_hold))
θ_all  = vcat(θ_data, fill(0.0, N_hold))
vy_all = vcat(vy_data, fill(vy_data[end], N_hold))
T_all  = vcat(T_data, fill(0.0, N_hold))
t_all  = vcat(collect(t_sim), fill(t_sim[end], N_hold))
N_total = length(x_all)

In [ ]:
# ============================================================
# Animate: tracking camera + live telemetry
# ============================================================

fig = Figure(size=(1400, 800))

# LEFT: trajectory with tracking camera
ax_traj = Axis(fig[1:3, 1], xlabel="Downrange (m)", ylabel="Altitude (m)",
    title="Falcon 9 Landing Burn")
hlines!(ax_traj, [0.0], color=:grey50, linewidth=2)
scatter!(ax_traj, [0.0], [0.0], color=:red, markersize=20, marker=:star5, label="Pad")

trail = Observable(Point2f[])
lines!(ax_traj, trail, color=(:dodgerblue, 0.5), linewidth=2, label="Trajectory")
rocket_pos = Observable([Point2f(x_data[1], y_data[1])])
scatter!(ax_traj, rocket_pos, color=:white, markersize=18, strokecolor=:black, strokewidth=2)
arrow_base = Observable([Point2f(x_data[1], y_data[1])])
arrow_dir  = Observable([Vec2f(0, 30)])
arrows!(ax_traj, arrow_base, arrow_dir, color=:orange, linewidth=3, arrowsize=12)
axislegend(ax_traj, position=:lt)

# RIGHT: telemetry
ax_alt = Axis(fig[1, 2], ylabel="Altitude (m)", title="Telemetry")
alt_line = Observable(Point2f[])
lines!(ax_alt, alt_line, color=:dodgerblue, linewidth=2)
hlines!(ax_alt, [0.0], color=:red, linestyle=:dash, linewidth=1)

ax_vel = Axis(fig[2, 2], ylabel="Vert. Velocity (m/s)")
vel_line = Observable(Point2f[])
lines!(ax_vel, vel_line, color=:red, linewidth=2)
hlines!(ax_vel, [0.0], color=:black, linestyle=:dash, linewidth=1)

ax_thr = Axis(fig[3, 2], xlabel="Time (s)", ylabel="Thrust (kN)")
thr_line = Observable(Point2f[])
lines!(ax_thr, thr_line, color=:orange, linewidth=2)
hlines!(ax_thr, [T_MAX/1000], color=:grey50, linestyle=:dash, linewidth=1)

t_end = t_sim[end]
limits!(ax_alt, 0, t_end*1.05, -20, maximum(y_data)*1.05)
limits!(ax_vel, 0, t_end*1.05, minimum(vy_data)*1.1, 5)
limits!(ax_thr, 0, t_end*1.05, -10, T_MAX/1000*1.1)

status_text = Observable("")
text!(ax_traj, 0.02, 0.98, text=status_text, space=:relative, align=(:left, :top),
      fontsize=16, color=:grey30)

record(fig, "falcon9_landing.mp4", 1:N_total; framerate=30) do i
    xi, yi, θi = x_all[i], y_all[i], θ_all[i]
    vyi, Ti, ti = vy_all[i], T_all[i], t_all[i]

    rocket_pos[] = [Point2f(xi, yi)]
    arrow_base[] = [Point2f(xi, yi)]
    arrow_dir[]  = [Vec2f(-30.0*sin(θi), 30.0*cos(θi))]
    push!(trail[], Point2f(xi, yi)); notify(trail)

    # Tracking camera
    cm_x = max(40.0, yi * 0.15)
    cm_y = max(50.0, yi * 0.20)
    limits!(ax_traj, min(xi,0)-cm_x, max(xi,0)+cm_x, min(yi,0)-20, yi+cm_y)

    if i <= N_sim
        push!(alt_line[], Point2f(ti, yi)); notify(alt_line)
        push!(vel_line[], Point2f(ti, vyi)); notify(vel_line)
        push!(thr_line[], Point2f(ti, Ti/1000)); notify(thr_line)
        status_text[] = "t = $(round(ti,digits=1)) s\ny = $(round(yi,digits=1)) m\nvy = $(round(vyi,digits=1)) m/s"
    else
        fuel = round(M0 - sol_s3[m_s3][end], digits=0)
        status_text[] = "✓ LANDED\nt = $(round(ti,digits=1)) s\nvy = $(round(vyi,digits=2)) m/s\nFuel: $(Int(fuel))/$(Int(M_FUEL)) kg"
    end
end

println("Animation saved to falcon9_landing.mp4")

---
## What We Built and What's Left

**What we modeled:**
- 2D rigid body dynamics (position, angle, velocities, spin)
- Thrust vector control via engine gimbal
- PD attitude control with smooth saturation
- Velocity-profile guided descent (suicide burn)
- Lateral steering to hit the pad
- Mass depletion from fuel burn

**What real Falcon 9 adds:**
- Aerodynamic drag and grid fins
- 3D dynamics (roll, pitch, yaw)
- Entry burn (3-engine) before landing burn (1-engine)
- The 40% minimum throttle constraint
- GPS + inertial navigation
- Convex optimization for fuel-optimal trajectories
- Structural flex, propellant sloshing, wind gusts